# `agg_df` — Advanced Reference

`agg_df` auto-detects group columns (non-numeric) and aggregates all numeric columns.

| `aggfunc` type | Behaviour |
|---|---|
| `str` e.g. `'sum'` | One aggregation on all numeric cols |
| `list` e.g. `['mean','n']` | Multiple aggregations; `n` always placed first |
| `dict` e.g. `{'col':'mean','cnt':'n'}` | Per-column control; column order follows dict key order |

`n` is a special token for group count — never an actual column name in the source data.

---

In [ ]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']

ModuleNotFoundError: No module named 'pytae'

## List aggfunc — `n` always goes first

In [ ]:
# mean + n: regardless of list order, n always comes first before aggregated columns
(penguins
 .select(['species', 'island', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])  # notice n comes first
)

Maddy


,species,island,n,bill_length_mm,body_mass_g
0,Adelie,Biscoe,44,38.975000,3709.659091
1,Adelie,Dream,56,38.501786,3688.392857
2,Adelie,Torgersen,52,38.950980,3706.372549
3,Chinstrap,Dream,68,48.833824,3733.088235
4,Gentoo,Biscoe,124,47.504878,5076.016260


In [ ]:
# Multiple numeric aggregations — column names get suffix: col_sum, col_mean, col_max
# also works with a single string if you only need one aggfunc
(penguins
 .select(['species', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['sum', 'mean', 'max', 'n'])
)

Maddy


,species,n,bill_length_mm_sum,bill_length_mm_mean,bill_length_mm_max,body_mass_g_sum,body_mass_g_mean,body_mass_g_max
0,Adelie,152,5857.5,38.791391,46.0,558800.0,3700.662252,4775.0
1,Chinstrap,68,3320.7,48.833824,58.0,253850.0,3733.088235,4800.0
2,Gentoo,124,5843.1,47.504878,59.6,624350.0,5076.016260,6300.0


## Dict aggfunc — per-column control, output order follows dict keys

In [ ]:
# Dict: mean for bill_length, sum+mean for body_mass, count named 'n'
# Output columns follow dict key order after group columns
(penguins
 .select(['species', 'island', 'bill_length_mm', 'body_mass_g'])
 .agg_df({'bill_length_mm': 'mean', 'body_mass_g': ['sum', 'mean'], 'n': 'n'})
)

Maddy


,species,island,bill_length_mm,body_mass_g_sum,body_mass_g_mean,n
0,Adelie,Biscoe,38.975000,163225.0,3709.659091,44
1,Adelie,Dream,38.501786,206550.0,3688.392857,56
2,Adelie,Torgersen,38.950980,189025.0,3706.372549,52
3,Chinstrap,Dream,48.833824,253850.0,3733.088235,68
4,Gentoo,Biscoe,47.504878,624350.0,5076.016260,124


In [ ]:
# Dict: place count first (n at front), then aggregations
# Unlike list aggfunc, with dict you control where n appears
# Note: the count key can be named anything — 'cnt', 'count', even 'maddy'
(tips
 .select(['smoker', 'time', 'total_bill', 'tip'])
 .agg_df({'cnt': 'n', 'total_bill': 'mean', 'tip': ['mean', 'max']})
)

Maddy


,smoker,time,cnt,total_bill,tip_mean,tip_max
0,Yes,Lunch,23,17.399130,2.834348,5.0
1,Yes,Dinner,70,21.859429,3.066000,10.0
2,No,Lunch,45,17.050889,2.673778,6.7
3,No,Dinner,106,20.095660,3.126887,9.0


In [ ]:
# You can name the count column anything you like in the dict
# Here 'maddy': 'n' creates a column called 'maddy' that holds the group count
(penguins
 .select(['species', 'island', 'body_mass_g'])
 .agg_df({'maddy': 'n', 'body_mass_g': 'sum'})
)

Maddy


,species,island,maddy,body_mass_g
0,Adelie,Biscoe,44,163225.0
1,Adelie,Dream,56,206550.0
2,Adelie,Torgersen,52,189025.0
3,Chinstrap,Dream,68,253850.0
4,Gentoo,Biscoe,124,624350.0


In [ ]:
# Multiple count columns in one dict — dict keys must be unique, so each count gets a different name
(penguins
 .select(['species', 'island', 'body_mass_g'])
 .agg_df({'count': 'n', 'countv2': 'n', 'body_mass_g': 'mean'})
)

Maddy


,species,island,count,countv2,body_mass_g
0,Adelie,Biscoe,44,44,3709.659091
1,Adelie,Dream,56,56,3688.392857
2,Adelie,Torgersen,52,52,3706.372549
3,Chinstrap,Dream,68,68,3733.088235
4,Gentoo,Biscoe,124,124,5076.016260


In [ ]:
# median is a standard aggfunc — works in list form and dict form
(penguins
 .select(['species', 'island', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['median', 'mean', 'n'])
)

Maddy


,species,island,n,bill_length_mm_median,bill_length_mm_mean,body_mass_g_median,body_mass_g_mean
0,Adelie,Biscoe,44,38.70,38.975000,3750.0,3709.659091
1,Adelie,Dream,56,38.55,38.501786,3575.0,3688.392857
2,Adelie,Torgersen,52,38.90,38.950980,3700.0,3706.372549
3,Chinstrap,Dream,68,49.55,48.833824,3700.0,3733.088235
4,Gentoo,Biscoe,124,47.30,47.504878,5000.0,5076.016260


## `dropna=False` — include groups with NaN keys

In [ ]:
# Penguins has NaN in 'sex' — dropna=False keeps those rows in their own group
(penguins
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'], dropna=False)
)

Maddy


,species,sex,n,bill_length_mm,body_mass_g
0,Adelie,Female,73,37.257534,3368.835616
1,Adelie,Male,73,40.390411,4043.493151
2,Adelie,NaN,6,37.840000,3540.000000
3,Chinstrap,Female,34,46.573529,3527.205882
4,Chinstrap,Male,34,51.094118,3938.970588
5,Gentoo,Female,58,45.563793,4679.741379
6,Gentoo,Male,61,49.473770,5484.836066
7,Gentoo,NaN,5,45.625000,4587.500000


## `n`-only aggregation — count with no numeric columns needed

In [ ]:
# Count only — useful for frequency tables on categorical columns
(penguins
 .select(exclude_dtype='number')
 .agg_df(aggfunc='n')
)

Maddy


,species,island,sex,n
0,Adelie,Biscoe,Female,22
1,Adelie,Biscoe,Male,22
2,Adelie,Dream,Female,27
3,Adelie,Dream,Male,28
4,Adelie,Torgersen,Female,24
5,Adelie,Torgersen,Male,23
6,Chinstrap,Dream,Female,34
7,Chinstrap,Dream,Male,34
8,Gentoo,Biscoe,Female,58
9,Gentoo,Biscoe,Male,61
